## Transferring Data from Device manufacturer sites to Influxdb
For flexibility and easy of data manipulation, we transfer the data to influx db.
This will also enable us to have the data downloaded and stored in one place - giving us ownership incase our subscription with the device manufacturer expires.

### Prepare tools for the project
First install the packages and modules necessary to implement the steps of the project.

The documentation for the influxdb_client_3 library can be found [here](https://docs.influxdata.com/influxdb3/cloud-dedicated/reference/client-libraries/v3/python/)

In [18]:
#Install operational packages to help in 
#organizing and querrying the data 
import pandas as pd
import numpy as np
import requests 
import json

#Modules to help interact with influxdb
import os, time
from influxdb_client_3 import InfluxDBClient3, Point


Create a magic function to help in skipping cells so that they run only once. Some cells especially those that involve uplaoding data to the database, can be skipped so that the data is not duplicated when the cells are are run multiple times

In [19]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

### Set up influxdb
Prepare the database to receive data from the device manufacturer urls 

In [20]:
#add this to load and upadte the token file
%reload_ext dotenv
%dotenv

#Define some of the arguments required 
#to write the data to the database

token = os.environ.get("INFLUXDB_TOKEN")
org         =   "Air quality - Harrisburg and Philadelphia"
host        =   "https://us-east-1-1.aws.cloud2.influxdata.com"
database    =   "pilot-harrisburg-home"

client = InfluxDBClient3(host=host, token=token, org=org, database=database)

### Upload the data from Devices to influxdb
Make API calls to the device urls and send the data to the influxdb database. Since different manufacturers have different procedures for API calls, we do upload the data in different batches based on the manufacturer

1. QUANTAQ

The guidelines to make API calls to quantaq can be found [here](https://docs.quant-aq.com/software-apis-and-libraries/quantaq-cloud-api)

In [21]:
#Load the api key and device serial numbers from the envrionment variable 
%reload_ext dotenv
%dotenv
QUANTAQ_APIKEY = os.environ.get("QUANTAQ_APIKEY")
QUANT001_SN = os.environ.get("QUANT001_SN")
QUANT002_SN = os.environ.get("QUANT002_SN")
CLARITY_APIKEY = os.environ.get("CLARITY_APIKEY")
CLARITY_ORGID = os.environ.get("CLARITY_ORGID")


In [5]:
def get_quantaq_data(url_ending):
    """
    A function that takes a list of words that go immediately after
    after the "https://api.quant-aq.com" of the url and help to 
    get access to scrap data from quantaq

    Args: 
        url_list (list):    A list of strings to be added to the url 
    
    Returns:
        response_dict (dictionary): a dictionary with the requested data        
    """

    #Define the fist part of the url
    url_first_part = "https://api.quant-aq.com/"

    #Initialize a continieoulsy increasing string to be used to complete the url
    url_additional = ""

    #Complete the url with user input strings 
    for adds_on in url_ending:
        url_additional = url_additional+adds_on+"/"        
    complete_url = url_first_part+url_additional

    #Make the call of the response to be returned 
    response_json = requests.get(complete_url, auth=(QUANTAQ_APIKEY,""))

    #Convert to dictionary for easy manipulation in python
    response_dict = json.loads(response_json.content)

    return response_dict

In [80]:
def upload_to_inlfuxdb(data_frame_to_save, measurement_name, index_field, client_name):
    """
    A function that takes the name of the measurement(table) 
    and one field name to serve as the index and wites them into
    a defined database on influxdb

    Args:
        data_frame_to_save(df)  : The dataframe to be sent to the database
        measurement_name(string) : The measurement or table name to store the dataset
        index_field(string)     : A timefield from the measurement to be used as an index
                                This also serves as the timeseries reference column
        client_name(object)     : The influxdb object with a defined database 

    Return : 
            Prints that it completed the task
    """ 

    #Make the index field to be the index so that it acts as the timestamp
    data_frame_to_save_index = data_frame_to_save.set_index(index_field)
    
    #Write to the database and the let user know that it is written
    client_name.write(record=data_frame_to_save_index, 
                      data_frame_measurement_name=measurement_name)
    
    print("Done! Check your influxdb to confirm!")



Transform the quantaaq data obtained through the API into a datframe, and send it to the influxdb database for further querrying and vizualization

In [47]:
#Get the json data to be saved to influxdb from the quantaq
data_meta_quant002_dict = get_quantaq_data(['v1','devices',QUANT002_SN,'data'])

#Extract the data part of the transformed data 
# for easy saving to the influxdb database
data_quant002_dict = data_meta_quant002_dict['data']

#Transform the data to a dataframe for 
# easy compatibility with influxdb classes
data_quant002_df = pd.DataFrame(data=data_quant002_dict)

#Use the upload function to send this datframe to inlflux
upload_to_inlfuxdb(data_frame_to_save=data_quant002_df, 
                   measurement_name='quantaq-pilot', 
                   index_field='timestamp', 
                   client_name=client)

Done! Check your influxdb to confrim!


In [ ]:
#Get the json data to be saved to influxdb from the quantaq
data_meta_quant002_dict = get_quantaq_data(['v1','devices',QUANT002_SN,'data'])

#Extract the data part of the transformed data 
# for easy saving to the influxdb database
data_quant002_dict = data_meta_quant002_dict['data']

#Transform the data to a dataframe for 
# easy compatibility with influxdb classes
data_quant002_df = pd.DataFrame(data=data_quant002_dict)

#Set the UTC timestamp to be the index for direct querrying
data_quant002_df_index = data_quant002_df.set_index('timestamp')

#Define the table name where the data will be saved in influxsb
measurement_name_quantaq = "quantaq-pilot"

#Save the data to influxdb to allow querying and future manipulation
client.write(record=data_quant002_df_index, 
             data_frame_measurement_name = measurement_name_quantaq)

#### 2. Clarity
The general guidelines for accessing clarity data through the api can be found [here](https://api-guide.clarity.io/); These guideline use examples based on POSTMAN - a webtool for obtaining data using APIs. Clarity has a python library for retrieving data found [here](https://github.com/a2gov/clarityio). Here we use the python library.

First install the clarity library

In [8]:
%%skip
pip install clarityio

Make initial connections with the clarity library


In [22]:
import clarityio # Offers methods to connect to the clarity api
from io import StringIO  # Helps to read file as csv and transform to df
api_connection = clarityio.ClarityAPIConnection(api_key=CLARITY_APIKEY, org=CLARITY_ORGID)

Getting the data from the APi before uploading to the database

In [72]:
def create_df_from_date(start_time):

    """
    A function that produces a clarity dataframe from a given to now

    Args:
        start_time(string) : Date and time from to start pulling records
                             In the format of '2026-01-04T00:00:00Z' 
    
    Return:
        latest_clarity_df (dataframe) : A dataframe with data from the given date to now

    """

    #Attemopt to get data from a certain date
    request_body_pilot = {
        'allDatasources': True,
        'outputFrequency': 'hour',
        'format':       'csv-wide',
        'startTime': start_time
        }

    #get the data in form of csv, ocnvert it to a dataframe and 
    # for easy manipulation and saving to the database
    response_wide2 = api_connection.get_recent_measurements(data=request_body_pilot)
    clarity_df = pd.read_csv(StringIO(response_wide2), sep=",")

    #Ensure that only meaningful columns exist to allow 
    # easy data search and manipulation in influxdb
    latest_clarity_df = clarity_df.dropna(axis='columns', how='all')

    return latest_clarity_df


Use the influxdb query package to get the latest date from the database. Look at the date and ensure that it makes sense. 

In [70]:

query_date = """SELECT *
                FROM  'clarity-pilot' 
                ORDER BY time DESC LIMIT 1
             """
    
#Use the queery on the influxdb database
table = client.query(query=query_date, language='sql')

#Obtain the date to use as a start date in the uplaoding 
# the most recent data from clarity to influxdb
most_recent_clarity_df = table.to_pandas()
latest_date = most_recent_clarity_df.startOfPeriod[0]

#verify that the date is in the right format and makes sense
print(latest_date)


2026-01-07T20:00:00Z


From the clarity API, get the data from the most recent date of saving in influxdb to now. And then make a dataframe from it which will later be added to the influxdb database

In [75]:
latest_clarity_df = create_df_from_date(start_time=latest_date)

Upload this dataframe to the influxdb database in the cloud. Ensure that the measurement name and client name in the uplaod function match with what was used to generate the dataframe

In [79]:
#Use the upload function to send this datframe to inlflux
upload_to_inlfuxdb(data_frame_to_save=latest_clarity_df, 
                   measurement_name='clarity-pilot', 
                   index_field='endOfPeriod', 
                   client_name=client)

Done! Check your influxdb to confrim!
